# 82 - U20-gated Q-gradient evaluation, worker 2/4

This worker runs shard 2 of the frozen held-out PRO160 position-perturbation cohort: 40 identities times two arms = 80 new rollouts. Both arms first measure stock P&P U20 with K=5 at zero-based Euler steps (3,4). They apply one latent Q-ascent update at Euler step 3 with RMS 0.005 only when U20 >= 0.0225; otherwise they execute the exact stock chunk returned by the measurement pass.

Periodic exact-matched tables print every 10 identities completed under both arms and include the matched historical stock-VLA outcome from notebook 68. The decoder uses 10 Euler steps, executes the first 10 of 50 generated actions, and replans. Videos, frames, and generated chunks are off.

## 1. Setup

In [ ]:
EXTRAS = 'sim'
SETUP_ENV = True
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

## 2. Download the two frozen Q50 checkpoints

In [ ]:
from pathlib import Path
from google.colab import userdata
from huggingface_hub import hf_hub_download

SHARD_COUNT = 4
SHARD_INDEX = 2
EPISODE_LIMIT = None  # set to 1 for a two-rollout smoke test
U20_GATE_THRESHOLD = 0.0225
LATENT_UPDATE_RMS = 0.005
HF_REPO_ID = 'Lilac2302/pnp-ckpts'
HF_REVISION = 'main'  # replace with an immutable repo commit SHA if available
HF_TOKEN = userdata.get('HF_TOKEN')
LOCAL_CHECKPOINT_DIR = Path('/content/qplanning_checkpoints')

def download(filename):
    return Path(hf_hub_download(
        repo_id=HF_REPO_ID,
        filename=filename,
        revision=HF_REVISION,
        token=HF_TOKEN,
        local_dir=LOCAL_CHECKPOINT_DIR,
    ))

ORIGINAL_Q50_CHECKPOINT_PATH = download('original_q50_step8000.pt')
U20_8CHUNK_CHECKPOINT_PATH = download('u20_8chunk_priority_q50_step6000.pt')

print({
    'worker': f'{SHARD_INDEX}/{SHARD_COUNT}',
    'identities': 40 if EPISODE_LIMIT is None else EPISODE_LIMIT,
    'new_rollouts': 80 if EPISODE_LIMIT is None else 2 * EPISODE_LIMIT,
    'arms': ['U20-gated original Q50', 'U20-gated 8-chunk-priority Q50'],
    'u20_gate_threshold': U20_GATE_THRESHOLD,
    'gate_measurement': 'K=5, Euler steps (3,4), first 20 actions',
    'q_gradient': 'Euler step 3, latent RMS 0.005',
    'decode_execute': '10 Euler integration steps; execute actions 0-9',
    'periodic_print_every_complete_identities': 10,
    'historical_stock': 'exact matched notebook-68 outcomes; not rerun',
    'checkpoints': {
        'original': str(ORIGINAL_Q50_CHECKPOINT_PATH),
        'u20_8chunk': str(U20_8CHUNK_CHECKPOINT_PATH),
    },
})

## 3. Run this shard's two gated Q-gradient arms

In [ ]:
from pnp.qplanning_u20_gated_gradient_eval_experiment import (
    QGUIDE_U20_GATE_EXPERIMENT,
    run_qplanning_u20_gated_gradient_heldout160,
)

report = run_qplanning_u20_gated_gradient_heldout160(
    original_checkpoint_path=ORIGINAL_Q50_CHECKPOINT_PATH,
    u20_8chunk_checkpoint_path=U20_8CHUNK_CHECKPOINT_PATH,
    shard_count=SHARD_COUNT,
    shard_index=SHARD_INDEX,
    episode_limit=EPISODE_LIMIT,
    threshold=U20_GATE_THRESHOLD,
    update_rms=LATENT_UPDATE_RMS,
    experiment=QGUIDE_U20_GATE_EXPERIMENT,
)
report

## 4. Audit persisted gate settings and telemetry

In [ ]:
from pnp.qplanning_u20_gated_gradient_eval_experiment import (
    validate_qplanning_u20_gated_gradient_sentinel,
)

validate_qplanning_u20_gated_gradient_sentinel(
    checkpoint_ids=report['checkpoint_ids'],
    experiment=report['experiment'],
)